<a href="https://colab.research.google.com/github/Timang419/deep-learning-for-mortgage-/blob/main/Full_fold_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install hdf5plugin --quiet
import os, pickle
import numpy as np
import pandas as pd
import h5py, hdf5plugin

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR  = '/content/drive/MyDrive/dissertation'
LOAN_HDF5 = os.path.join(BASE_DIR, 'hdf5_full_merged')

CHUNK_ROWS = 20_000_000

with h5py.File(os.path.join(LOAN_HDF5, 'test.h5'), 'r') as f:
    full_feat_cols = [c.decode('utf-8') if isinstance(c, bytes) else c
                      for c in f['X'].attrs['col_names']]
    N_TEST_ROWS = f['X'].shape[0]
print(f'test.h5 rows: {N_TEST_ROWS:,}')

with open(os.path.join(LOAN_HDF5, 'scaler.pkl'), 'rb') as f:
    loaded = pickle.load(f)
loan_scaler = loaded['scaler'] if isinstance(loaded, dict) else loaded

CONTINUOUS_COLS = [
    'Current Actual UPB', 'Current Loan Delinquency Status', 'Loan Age',
    'Remaining Months to Legal Maturity', 'Current Interest Rate',
    'Current Non-Interest Bearing UPB', 'Current Month Modification Cost',
    'Interest Bearing UPB', 'Months Since Paid', 'Cumulative_Mod_Cost_Engineered',
    'D30_count_12m', 'D60_count_12m', 'D90plus_count_12m', 'Credit Score',
    'Mortgage Insurance Percentage', 'Original Debt-to-Income (DTI) Ratio',
    'Original UPB', 'Original Loan-to-Value (LTV)', 'Original Interest Rate',
    'Original Loan Term', 'CLTV_clean',
]
def sc_mean_scale(col_name):
    k = CONTINUOUS_COLS.index(col_name)
    return float(loan_scaler.mean_[k]), float(loan_scaler.scale_[k])

mod_mean, mod_scale = sc_mean_scale('Current Month Modification Cost')
mod_idx  = full_feat_cols.index('Current Month Modification Cost')
rp_idx   = full_feat_cols.index('Reporting_Period_Int')

keep_loan_ids, keep_periods, keep_costs = [], [], []

with h5py.File(os.path.join(LOAN_HDF5, 'test.h5'), 'r') as f:
    loan_ids_ds = f['loan_ids']
    for start in range(0, N_TEST_ROWS, CHUNK_ROWS):
        end = min(start + CHUNK_ROWS, N_TEST_ROWS)
        X_chunk = f['X'][start:end]
        raw_mod = X_chunk[:, mod_idx] * mod_scale + mod_mean
        nz_mask = np.abs(raw_mod) > 1e-6
        if nz_mask.any():
            ids_chunk = loan_ids_ds.asstr()[start:end]
            keep_loan_ids.append(ids_chunk[nz_mask])
            keep_periods.append(X_chunk[nz_mask, rp_idx])
            keep_costs.append(raw_mod[nz_mask])
        print(f'  rows {end:>15,}/{N_TEST_ROWS:,} processed', flush=True)

df = pd.DataFrame({
    'loan_id': np.concatenate(keep_loan_ids),
    'period':  np.concatenate(keep_periods),
    'cost':    np.concatenate(keep_costs),
}).sort_values(['loan_id', 'period'])

# ── per-loan classification (FIXED: handles year-boundary correctly) ───────
def classify(group):
    periods = group['period'].values
    n = len(periods)
    if n == 1:
        return 'isolated_single_month'

    # convert YYYYMM -> a flat, continuously-increasing month index,
    # so consecutive calendar months always have a diff of exactly 1,
    # even across a year boundary (e.g. 202112 -> 202201)
    years     = (periods // 100).astype(int)
    months    = (periods % 100).astype(int)
    month_idx = years * 12 + months

    diffs = np.diff(month_idx)
    if np.all(diffs == 1):
        return 'fully_persistent_run'
    elif np.any(diffs == 1):
        return 'mixed_runs_and_gaps'
    else:
        return 'scattered_non_consecutive'

loan_classes = df.groupby('loan_id').apply(classify)
counts = loan_classes.value_counts()
pct    = (loan_classes.value_counts(normalize=True) * 100).round(2)

print('\n' + '='*70)
print('PER-LOAN CLASSIFICATION OF MODIFICATION-COST PATTERN')
print('='*70)
n_loans_total = len(loan_classes)
print(f'Total loans with any nonzero Modification Cost: {n_loans_total:,}\n')
for cls in ['isolated_single_month', 'fully_persistent_run',
            'mixed_runs_and_gaps', 'scattered_non_consecutive']:
    c = counts.get(cls, 0)
    p = pct.get(cls, 0.0)
    print(f'  {cls:<28s}: {c:>8,} loans  ({p:>5.2f}%)')

print('\nExamples of each category:')
for cls in ['isolated_single_month', 'fully_persistent_run',
            'mixed_runs_and_gaps', 'scattered_non_consecutive']:
    sample_ids = loan_classes[loan_classes == cls].index[:2]
    print(f'\n--- {cls} ---')
    for lid in sample_ids:
        sub = df[df['loan_id'] == lid][['period', 'cost']]
        print(f'  loan {lid}:')
        print(sub.to_string(index=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 46.2 MB/s eta 0:00:00
Mounted at /content/drive
test.h5 rows: 719,770,410


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  rows      20,000,000/719,770,410 processed
  rows      40,000,000/719,770,410 processed
  rows      60,000,000/719,770,410 processed
  rows      80,000,000/719,770,410 processed
  rows     100,000,000/719,770,410 processed
  rows     120,000,000/719,770,410 processed
  rows     140,000,000/719,770,410 processed
  rows     160,000,000/719,770,410 processed
  rows     180,000,000/719,770,410 processed
  rows     200,000,000/719,770,410 processed
  rows     220,000,000/719,770,410 processed
  rows     240,000,000/719,770,410 processed
  rows     260,000,000/719,770,410 processed
  rows     280,000,000/719,770,410 processed
  rows     300,000,000/719,770,410 processed
  rows     320,000,000/719,770,410 processed
  rows     340,000,000/719,770,410 processed
  rows     360,000,000/719,770,410 processed
  rows     380,000,000/719,770,410 processed
  rows     400,000,000/719,770,410 processed
  rows     420,000,000/719,770,410 processed
  rows     440,000,000/719,770,410 processed
  rows    

/tmp/ipykernel_9065/1777075208.py:87: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  loan_classes = df.groupby('loan_id').apply(classify)



PER-LOAN CLASSIFICATION OF MODIFICATION-COST PATTERN
Total loans with any nonzero Modification Cost: 543,093

  isolated_single_month       :   11,114 loans  ( 2.05%)
  fully_persistent_run        :  530,405 loans  (97.66%)
  mixed_runs_and_gaps         :    1,569 loans  ( 0.29%)
  scattered_non_consecutive   :        5 loans  ( 0.00%)

Examples of each category:

--- isolated_single_month ---
  loan F00Q30242692:
  period  cost
202101.0 -0.33
  loan F01Q10171593:
  period       cost
202101.0 101.040001

--- fully_persistent_run ---
  loan F00Q10010967:
  period  cost
202101.0  8.33
202102.0  8.33
202103.0  8.33
202104.0  8.33
202105.0  8.33
  loan F00Q10013028:
  period      cost
202101.0 20.790001
202102.0 20.790001
202103.0 20.790001
202104.0 20.790001
202105.0 20.790001
202106.0 20.790001
202107.0 20.790001
202108.0 20.790001
202109.0 20.790001
202110.0 20.790001
202111.0 20.790001
202112.0 20.790001
202201.0 20.790001
202202.0 20.790001
202203.0 20.790001
202204.0 20.790001
20220

In [ ]:
!pip install hdf5plugin optuna seaborn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 42.4 MB/s eta 0:00:00


In [ ]:
import os, gc, json, warnings, time, pickle
import threading
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import ctypes
import glob
import shutil

n_cpus = os.cpu_count()
os.environ['HDF5_PLUGIN_MAX_THREADS'] = str(n_cpus)

import hdf5plugin          # MUST import before h5py — registers LZ4 codec
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

from sklearn.metrics import (
    f1_score, accuracy_score, log_loss,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import StandardScaler

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore', category=UserWarning)


# DEVICE, SEED, AMP

In [ ]:
SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

if DEVICE.type == 'cuda':
    _cap      = torch.cuda.get_device_capability()
    AMP_DTYPE = torch.bfloat16 if _cap[0] >= 8 else torch.float16
    print(f'Device: {DEVICE}  ({torch.cuda.get_device_name(0)},  '
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB VRAM)')
else:
    AMP_DTYPE = torch.float16
    print(f'Device: {DEVICE}')

Device: cuda  (NVIDIA L4,  24 GB VRAM)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# HDF5_DIR   = '/mnt/local-scratch/merged'
# OUTPUT_DIR = '/content/drive/MyDrive/dissertation/models/rolling_test'
# SCRATCH    = '/mnt/local-scratch'
# DRIVE_HDF5 = '/content/drive/MyDrive/dissertation/hdf5_full_merged'
HDF5_DIR   = '/content/merged'
OUTPUT_DIR = '/content/drive/MyDrive/dissertation/models/rolling_test_sample_v2'
SCRATCH    = '/content'
DRIVE_HDF5 = '/content/drive/MyDrive/dissertation/hdf5_sample_fullmerged'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HDF5_DIR,   exist_ok=True)

# CONSTANTS


In [ ]:
NUM_CLASSES  = 7
STATE_NAMES  = ['Current', 'D30', 'D60', 'D90+', 'Foreclosure', 'REO', 'PaidOff']
REPORTING_PERIOD_COL = 'Reporting_Period_Int'

ZL_LR        = 1e-3    # zero-layer (logistic regression) — matches FIXED_LR in script3
MN_LR        = 1e-4    # MortgageNet — matches final training lr in script3
WD_GRID      = [1e-5, 1e-4, 1e-3, 1e-2]   # weight_decay grid, same for both models
MN_ARCH      = dict(depth=5, hidden_dim=512, dropout=0.2, norm_type='none')  # fixed from main run

# CHUNK_ROWS   = 60_000_000
# HPO_ROWS     = 60_000_000
# HPO_VAL_ROWS = 15_000_000
# SCALER_CHUNK_ROWS = 10_000_000
# HPO_EPOCHS   = 10
# FINAL_EPOCHS = 3

CHUNK_ROWS        = 5_000_000
HPO_ROWS          = 5_000_000
HPO_VAL_ROWS      = 1_500_000
SCALER_CHUNK_ROWS = 2_000_000
HPO_EPOCHS        = 10
FINAL_EPOCHS      = 10

BINARY_PREFIXES = (
    'state_', 'mod_flag_', 'step_mod_', 'deferral_', 'assist_',
    'FTHB_', 'UNITS_', 'OCC_', 'CHAN_', 'PROP_', 'PURP_',
    'BORR_', 'PROG_', 'VAL_', 'MI_CANCEL_', 'STATE_', 'VINT_',
)
BINARY_EXACT = {
    'MSA', 'Prepayment Penalty Mortgage (PPM) Flag',
    'Super Conforming Flag', 'HARP Indicator',
    'Interest Only (I/O) Indicator', 'MI_OUT_OF_RANGE',
    'CLTV_MISSING', 'DTI_MISSING', 'Delinquency Due to Disaster',
    'Reporting_Period_Int',
}

# COPY DATA TO NVMe

In [ ]:
def _copy_to_nvme(fname):
    src = os.path.join(DRIVE_HDF5, fname)
    dst = os.path.join(HDF5_DIR,   fname)
    if os.path.exists(dst):
        print(f'  {fname}: already on NVMe ({os.path.getsize(dst) / 1e9:.1f} GB)')
        return
    print(f'  copying {fname} ...', flush=True)
    t0  = time.time()
    shutil.copy2(src, dst)
    gb  = os.path.getsize(dst) / 1e9
    el  = max(time.time() - t0, 1e-3)
    print(f'  {fname}: done  {gb:.1f} GB  in  {el:.0f}s  '
          f'({gb * 1000 / el:.0f} MB/s)', flush=True)

print('\n=== Copying data to NVMe ===')
with ThreadPoolExecutor(max_workers=5) as _p:
    list(_p.map(_copy_to_nvme,
                [f'train_shard_{i}_merged.h5' for i in range(10)]))
_copy_to_nvme('feature_cols.json')
_copy_to_nvme('scaler.pkl')
#_copy_to_nvme('test_2021.h5')
print('=== NVMe copy done ===\n')


=== Copying data to NVMe ===
  copying train_shard_0_merged.h5 ...
  copying train_shard_1_merged.h5 ...
  copying train_shard_2_merged.h5 ...
  copying train_shard_3_merged.h5 ...
  copying train_shard_4_merged.h5 ...
  train_shard_2_merged.h5: done  0.8 GB  in  23s  (37 MB/s)
  copying train_shard_5_merged.h5 ...
  train_shard_0_merged.h5: done  0.8 GB  in  25s  (33 MB/s)
  copying train_shard_6_merged.h5 ...
  train_shard_1_merged.h5: done  0.9 GB  in  25s  (34 MB/s)
  copying train_shard_7_merged.h5 ...
  train_shard_4_merged.h5: done  0.8 GB  in  26s  (33 MB/s)
  copying train_shard_8_merged.h5 ...
  train_shard_3_merged.h5: done  0.8 GB  in  31s  (27 MB/s)
  copying train_shard_9_merged.h5 ...
  train_shard_6_merged.h5: done  0.8 GB  in  24s  (35 MB/s)
  train_shard_8_merged.h5: done  0.8 GB  in  24s  (35 MB/s)
  train_shard_5_merged.h5: done  0.8 GB  in  27s  (32 MB/s)
  train_shard_7_merged.h5: done  0.8 GB  in  24s  (33 MB/s)
  train_shard_9_merged.h5: done  0.8 GB  in  19s  

# RESOLVED PATHS

In [ ]:
N_MERGED     = 10
MERGED_PATHS = [os.path.join(HDF5_DIR, f'train_shard_{i}_merged.h5')
                for i in range(N_MERGED)]
TEST_H5_DRIVE = os.path.join(DRIVE_HDF5, 'test.h5')    # full test — Drive only
TEST_2021_NVME = os.path.join(HDF5_DIR, 'test_2021.h5')
COLS_JSON    = os.path.join(HDF5_DIR,   'feature_cols.json')
SCALER_PKL   = os.path.join(HDF5_DIR,   'scaler.pkl')
CKPT_FILE    = os.path.join(OUTPUT_DIR, 'rolling_checkpoint.json')

# FEATURE INDEX HELPERS

In [ ]:
def get_continuous_indices(col_names):
    return [i for i, col in enumerate(col_names)
            if not any(col.startswith(p) for p in BINARY_PREFIXES)
            and col not in BINARY_EXACT]

def get_model_feature_indices(feature_cols):
    return [i for i, c in enumerate(feature_cols) if c != REPORTING_PERIOD_COL]

def build_feature_maps(feature_cols):
    """
    Returns MODEL_FEAT_IDX, CONT_HDF5_IDX, CONT_FEAT_POS, RPI_ABS_IDX.

    CONT_FEAT_POS[k] = position of the k-th continuous column within MODEL_FEAT_IDX.
    Used for in-memory rescaling: X[:, CONT_FEAT_POS] = X[:, CONT_FEAT_POS] * a + b
    """
    MODEL_FEAT_IDX = get_model_feature_indices(feature_cols)
    CONT_HDF5_IDX  = np.array(get_continuous_indices(feature_cols))
    feat_pos_map   = {hdf5_idx: pos for pos, hdf5_idx in enumerate(MODEL_FEAT_IDX)}
    # Reporting_Period_Int is in BINARY_EXACT, so all continuous cols are in MODEL_FEAT_IDX
    CONT_FEAT_POS  = np.array([feat_pos_map[i] for i in CONT_HDF5_IDX])
    RPI_ABS_IDX    = feature_cols.index(REPORTING_PERIOD_COL)
    return MODEL_FEAT_IDX, CONT_HDF5_IDX, CONT_FEAT_POS, RPI_ABS_IDX

 FOLD DEFINITIONS

In [ ]:
def _s(paths, year_set=None):
    """Expand path(s) to list of (single_path, year_set) tuples."""
    if isinstance(paths, list):
        return [(p, year_set) for p in paths]
    return [(paths, year_set)]

# def _make_folds():
#     return [
#         dict(
#             name='fold1', test_label='2016-2018',
#             scaler_sources    = _s(MERGED_PATHS, set(range(1999, 2014))),
#             hpo_train_sources = _s(MERGED_PATHS, set(range(1999, 2014))),
#             hpo_val_sources   = _s(MERGED_PATHS, {2014, 2015}),
#             final_train_src   = _s(MERGED_PATHS, set(range(1999, 2016))),
#             test_sources      = _s(MERGED_PATHS, {2016, 2017, 2018}),
#         ),
#         dict(
#             name='fold2', test_label='2019-2021',
#             scaler_sources    = _s(MERGED_PATHS, set(range(1999, 2017))),
#             hpo_train_sources = _s(MERGED_PATHS, set(range(1999, 2017))),
#             hpo_val_sources   = _s(MERGED_PATHS, {2017, 2018}),
#             final_train_src   = _s(MERGED_PATHS, set(range(1999, 2019))),
#             test_sources      = _s(MERGED_PATHS,  {2019, 2020})
#                               + _s(TEST_2021_NVME, None),
#         ),
#         dict(
#             name='fold3', test_label='2022-2025',
#             scaler_sources    = _s(MERGED_PATHS, set(range(1999, 2020))),
#             hpo_train_sources = _s(MERGED_PATHS, set(range(1999, 2020))),
#             hpo_val_sources   = _s(MERGED_PATHS,  {2020})
#                               + _s(TEST_2021_NVME, None),
#             final_train_src   = _s(MERGED_PATHS,   None)
#                               + _s(TEST_2021_NVME, None),
#             test_sources      = _s(TEST_H5_DRIVE, set(range(2022, 2026))),
#         ),
#     ]

def _make_folds():
    return [
        dict(
            name='fold3', test_label='2022-2025',
            scaler_sources    = _s(MERGED_PATHS, set(range(1999, 2020))),
            hpo_train_sources = _s(MERGED_PATHS, set(range(1999, 2020))),
            hpo_val_sources   = _s(MERGED_PATHS,  {2020})
                              + _s(TEST_H5_DRIVE, {2021}),
            final_train_src   = _s(MERGED_PATHS,   None)
                              + _s(TEST_H5_DRIVE, {2021}),
            test_sources      = _s(TEST_H5_DRIVE, set(range(2022, 2026))),
        ),
    ]


In [ ]:
def fit_fold_scaler(sources, cont_hdf5_idx, rpi_abs_idx, existing_scaler,
                    n_workers=5):
    """
    Fit a new StandardScaler on fold-train continuous columns.
    Parallelises across source files using Chan's parallel algorithm —
    each worker computes local (n, mean, M2) then combined globally.
    No partial_fit — pure numpy, thread-safe (each thread opens its own h5py handle).
    Optimised: extract 21 continuous cols before applying boolean mask
    (reduces mask copy from 7GB to 840MB per chunk).
    """

    def _shard_stats(args):
        h5_path, year_set = args
        n_acc    = 0
        mean_acc = None
        M2_acc   = None

        with h5py.File(h5_path, 'r') as f:
            n_rows = f['X'].shape[0]

            for start in range(0, n_rows, SCALER_CHUNK_ROWS):
                end   = min(start + SCALER_CHUNK_ROWS, n_rows)
                X_raw = f['X'][start:end]

                if year_set is not None:
                    rpi  = (X_raw[:, rpi_abs_idx].astype(np.int64)) // 100
                    mask = np.isin(rpi, np.array(sorted(year_set)))
                    # Slice 21 cols BEFORE applying mask — 7GB → 840MB copy
                    X_cont = X_raw[:, cont_hdf5_idx]
                    del X_raw
                    if not mask.any():
                        del X_cont, mask; continue
                    X_cont = X_cont[mask].astype(np.float64)
                    del mask
                else:
                    X_cont = X_raw[:, cont_hdf5_idx].astype(np.float64)
                    del X_raw

                if len(X_cont) == 0:
                    del X_cont; continue

                X_cont = X_cont * existing_scaler.scale_ + existing_scaler.mean_

                n_chunk    = len(X_cont)
                mean_chunk = X_cont.mean(axis=0)
                M2_chunk   = ((X_cont - mean_chunk) ** 2).sum(axis=0)
                del X_cont

                if mean_acc is None:
                    n_acc = n_chunk; mean_acc = mean_chunk; M2_acc = M2_chunk
                else:
                    n_new    = n_acc + n_chunk
                    delta    = mean_chunk - mean_acc
                    mean_acc = (n_acc * mean_acc + n_chunk * mean_chunk) / n_new
                    M2_acc   = M2_acc + M2_chunk + delta**2 * n_acc * n_chunk / n_new
                    n_acc    = n_new

        return h5_path, n_acc, mean_acc, M2_acc

    # Parallelise across shards
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = list(pool.map(_shard_stats, sources))

    # Combine shard statistics with Chan's parallel combination
    n_total = 0; mean_total = None; M2_total = None

    for h5_path, n_s, mean_s, M2_s in futures:
        if n_s == 0 or mean_s is None:
            print(f'  [scaler] {os.path.basename(h5_path)}: 0 rows (filtered out)',
                  flush=True)
            continue
        print(f'  [scaler] {os.path.basename(h5_path)}: {n_s:,} rows', flush=True)
        if mean_total is None:
            n_total = n_s; mean_total = mean_s; M2_total = M2_s
        else:
            n_new      = n_total + n_s
            delta      = mean_s - mean_total
            mean_total = (n_total * mean_total + n_s * mean_s) / n_new
            M2_total   = M2_total + M2_s + delta**2 * n_total * n_s / n_new
            n_total    = n_new

    # Build StandardScaler from computed global statistics
    new_scaler                 = StandardScaler()
    new_scaler.mean_           = mean_total
    new_scaler.var_            = M2_total / n_total
    new_scaler.scale_          = np.where(new_scaler.var_ == 0, 1.0,
                                           np.sqrt(new_scaler.var_))
    new_scaler.n_samples_seen_ = n_total
    new_scaler.n_features_in_  = len(cont_hdf5_idx)

    print(f'  [scaler] combined: {n_total:,} total rows  '
          f'({n_workers} parallel workers)', flush=True)
    return new_scaler


def compute_rescale_coeffs(existing_scaler, new_scaler):
    """
    Compute (a, b) such that x_new_std = x_old_std * a + b.

      x_raw     = x_old_std * sigma_old + mu_old
      x_new_std = (x_raw - mu_new) / sigma_new
                = x_old_std * (sigma_old/sigma_new) + (mu_old - mu_new)/sigma_new
    """
    a = (existing_scaler.scale_ / new_scaler.scale_).astype(np.float32)
    b = ((existing_scaler.mean_ - new_scaler.mean_) / new_scaler.scale_).astype(np.float32)
    return a, b

# HPO SAMPLE LOADER

In [ ]:
def load_hpo_sample(sources, feat_indices, rpi_abs_idx,
                    rescale_a, rescale_b, cont_feat_pos, max_rows):
    """
    Stream sources, apply year filter + rescaling, accumulate up to max_rows.
    Returns (X float32, y int64) numpy arrays ready to push to VRAM.
    """
    X_acc = []; y_acc = []
    n_total = 0

    for h5_path, year_set in sources:
        if n_total >= max_rows:
            break

        with h5py.File(h5_path, 'r') as f:
            n_file = f['X'].shape[0]

        for start in range(0, n_file, CHUNK_ROWS):
            if n_total >= max_rows:
                break
            end = min(start + CHUNK_ROWS, n_file)

            with h5py.File(h5_path, 'r') as f:
                X_raw = f['X'][start:end]
                y_raw = f['y'][start:end].astype(np.int64)

            if year_set is not None:
                rpi  = (X_raw[:, rpi_abs_idx].astype(np.int64)) // 100
                mask = np.isin(rpi, np.array(sorted(year_set)))
                X_raw = X_raw[mask]
                y_raw = y_raw[mask]

            if len(y_raw) == 0:
                del X_raw, y_raw; continue

            take  = min(len(y_raw), max_rows - n_total)
            X_raw = X_raw[:take]
            y_raw = y_raw[:take]

            # Extract model features
            X = X_raw[:, feat_indices].astype(np.float32)
            del X_raw                               # crucial RAM saver
            # Apply linear rescaling to continuous columns
            X[:, cont_feat_pos] = (
                X[:, cont_feat_pos].astype(np.float64) * rescale_a + rescale_b
            ).astype(np.float32)

            X_acc.append(X); y_acc.append(y_raw)
            n_total += len(y_raw)
            del X, y_raw; gc.collect()

    print(f'  HPO sample: {n_total:,} rows loaded from '
          f'{len(X_acc)} chunks', flush=True)
    return np.concatenate(X_acc), np.concatenate(y_acc)

# FILTERED CHUNK PREFETCHER  (background thread, hides I/O behind GPU compute)

In [ ]:
class FilteredChunkPrefetcher:
    """
    True parallel prefetcher — reads, filters, and rescales inside each thread.
    Eliminates the serial 41.7GB boolean mask bottleneck that caused ~50s penalty.
    Each thread handles its own 5M-row sub-slice independently.
    """
    def __init__(self, feat_indices, rpi_abs_idx, rescale_a, rescale_b, cont_feat_pos):
        self._feat    = feat_indices
        self._rpi_idx = rpi_abs_idx
        self._a       = rescale_a.astype(np.float32)
        self._b       = rescale_b.astype(np.float32)
        self._cp      = cont_feat_pos
        self._result  = None
        self._thread  = None
        self._exc     = None

    def start(self, path, start, end, year_set):
        self._result = self._exc = None
        feat      = self._feat
        rpi_idx   = self._rpi_idx
        a, b, cp  = self._a, self._b, self._cp
        n_threads = int(os.environ.get('HDF5_PLUGIN_MAX_THREADS', str(os.cpu_count())))
        n_rows    = end - start

        bounds = np.linspace(0, n_rows, n_threads + 1, dtype=int)
        ranges = [(int(bounds[i]), int(bounds[i+1]))
                  for i in range(n_threads) if bounds[i] < bounds[i+1]]

        # Compute once outside threads — avoids repeated sorting
        valid_years = np.array(sorted(year_set)) if year_set is not None else None

        def _work():
            try:
                def _read_and_process(r_start, r_end):
                    abs_s = start + r_start
                    abs_e = start + r_end
                    with h5py.File(path, 'r') as f:
                        X_raw = f['X'][abs_s:abs_e]
                        y_raw = f['y'][abs_s:abs_e].astype(np.int64)

                    # Year filter inside thread — on 3.5GB not 41.7GB
                    if valid_years is not None:
                        rpi  = (X_raw[:, rpi_idx].astype(np.int64)) // 100
                        mask = np.isin(rpi, valid_years)
                        if not mask.any():
                            return None
                        X_raw = X_raw[mask]
                        y_raw = y_raw[mask]

                    # Feature extraction + rescaling inside thread
                    X_chunk = X_raw[:, feat].astype(np.float32)
                    del X_raw
                    X_chunk[:, cp] = X_chunk[:, cp] * a + b
                    return X_chunk, y_raw

                with ThreadPoolExecutor(max_workers=n_threads) as pool:
                    results = list(pool.map(lambda r: _read_and_process(*r), ranges))

                valid = [r for r in results if r is not None]

                if not valid:
                    self._result = (
                        np.empty((0, len(feat)), dtype=np.float32),
                        np.empty(0, dtype=np.int64))
                else:
                    X_out = np.concatenate([r[0] for r in valid], axis=0)
                    y_out = np.concatenate([r[1] for r in valid], axis=0)
                    self._result = (X_out, y_out)

            except Exception as exc:
                self._exc = exc

        self._thread = threading.Thread(target=_work, daemon=True)
        self._thread.start()

    def get(self):
        if self._thread is not None:
            self._thread.join()
        if self._exc is not None:
            raise self._exc
        return self._result

# MODEL

In [ ]:
class MortgageNet(nn.Module):
    """
    Feed-forward NN.
    depth=0 → single linear layer (≡ multinomial logistic regression, Model 2)
    depth>0 → MortgageNet (Model 3)
    """
    def __init__(self, input_dim, num_classes=NUM_CLASSES,
                 depth=5, hidden_dim=256, dropout_rate=0.3, norm_type='batch'):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(depth):
            layers.append(nn.Linear(d, hidden_dim))
            if norm_type == 'batch':
                layers.append(nn.BatchNorm1d(hidden_dim))
            elif norm_type == 'layer':
                layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_rate))
            d = hidden_dim
        self.hidden_layers = nn.Sequential(*layers)
        self.output_layer  = nn.Linear(d, num_classes)

    def forward(self, x):
        return self.output_layer(self.hidden_layers(x))

def make_optimizer(model, lr, weight_decay):
    decay    = [p for p in model.parameters() if p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.ndim  < 2]
    return torch.optim.AdamW(
        [{'params': decay,    'weight_decay': weight_decay},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=lr,
    )


# GPU TENSOR EVAL  (used inside HPO training for val loss per epoch)

In [ ]:
@torch.no_grad()
def _eval_gpu_tensors(model, X, y, crit, bs=32768):
    model.eval()
    total_loss = correct = 0
    all_preds  = []
    n = X.shape[0]
    for i in range(0, n, bs):
        X_b = X[i:i+bs]; y_b = y[i:i+bs]
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE,
                            enabled=DEVICE.type == 'cuda'):
            logits = model(X_b)
            loss   = crit(logits, y_b)
        logits_f = logits.float()
        preds    = logits_f.argmax(dim=1)
        total_loss += loss.item() * len(y_b)
        correct    += (preds == y_b).sum().item()
        all_preds.append(preds.cpu())
    f1 = f1_score(y.cpu().numpy(), torch.cat(all_preds).numpy(),
                  average='macro', zero_division=0)
    return total_loss / n, correct / n, f1

# HPO TRAINING  (in-VRAM, no early stopping, best state tracked in RAM)

In [ ]:
def train_nn(cfg, X_tr, y_tr, X_val, y_val, verbose=False):
    """
    HPO training on in-VRAM GPU tensors.
    Runs exactly HPO_EPOCHS epochs — no early stopping.
    Best val checkpoint kept in CPU RAM (no disk I/O per epoch).
    Returns (best_model, history).
    """
    n_tr = X_tr.shape[0]
    bs   = cfg['batch_size']

    model = MortgageNet(
        input_dim    = X_tr.shape[1],
        depth        = cfg['depth'],
        hidden_dim   = cfg['hidden_dim'],
        dropout_rate = cfg['dropout'],
        norm_type    = cfg['norm_type'],
    ).to(DEVICE)
    opt  = make_optimizer(model, cfg['lr'], cfg['weight_decay'])
    crit = nn.CrossEntropyLoss()

    best_val   = np.inf
    best_state = None          # CPU copy of best weights — no disk I/O
    history    = {'tr_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(1, HPO_EPOCHS + 1):
        model.train()
        perm           = torch.randperm(n_tr, device=DEVICE)
        total, n_total = 0.0, 0
        for i in range(0, n_tr - bs + 1, bs):
            idx = perm[i:i+bs]
            X_b = X_tr[idx]; y_b = y_tr[idx]
            opt.zero_grad()
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE,
                                enabled=DEVICE.type == 'cuda'):
                logits = model(X_b)
                loss   = crit(logits, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total   += loss.item() * bs
            n_total += bs
        tr_loss = total / n_total

        vl, va, vf1 = _eval_gpu_tensors(model, X_val, y_val, crit)
        history['tr_loss'].append(tr_loss)
        history['val_loss'].append(vl)
        history['val_acc'].append(va)
        history['val_f1'].append(vf1)

        if verbose:
            print(f'  Ep {epoch:3d}/{HPO_EPOCHS} | '
                  f'tr {tr_loss:.4f} | val {vl:.4f} | '
                  f'acc {va:.4f} | F1 {vf1:.4f}')

        # Track best val checkpoint in CPU RAM — zero disk I/O
        if vl < best_val:
            best_val   = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restore best weights before returning
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

# HPO GRID SEARCH  (both ZL and MN use this same function)

In [ ]:
def run_hpo_grid(model_name, base_cfg, X_tr_gpu, y_tr_gpu,
                 X_val_gpu, y_val_gpu, wd_grid, fold_dir):
    """
    Grid search over weight_decay for a fixed model architecture (base_cfg).
    HPO data stays in VRAM across all trials (pushed once, reused — matches script3).
    Each trial model is deleted after recording val_loss (VRAM freed between trials).
    Returns best weight_decay.
    """
    tag  = model_name.lower().replace(' ', '_')
    rows = []

    print(f'  [{model_name}] HPO: {len(wd_grid)} trials over WD_GRID {wd_grid}')

    for wd in wd_grid:
        cfg = dict(**base_cfg,
                   weight_decay = wd,
                   batch_size   = 16384)

        print(f'    wd={wd:.0e} ...', end=' ', flush=True)

        _model, hist = train_nn(cfg, X_tr_gpu, y_tr_gpu,
                                X_val_gpu, y_val_gpu,
                                verbose=False)

        best_val = min(hist['val_loss'])
        last_val = hist['val_loss'][-1]
        best_ep  = hist['val_loss'].index(best_val) + 1
        print(f'best_val={best_val:.5f}  last_val={last_val:.5f}  '
              f'best_ep={best_ep}', flush=True)
        rows.append(dict(weight_decay=wd, val_loss=best_val,
                         last_val=last_val, best_epoch=best_ep))

        # Free model VRAM between trials (HPO data stays in VRAM)
        del _model
        gc.collect(); torch.cuda.empty_cache()

    df      = pd.DataFrame(rows).sort_values('val_loss')
    best_wd = float(df.iloc[0]['weight_decay'])
    print(f'\n  [{model_name}] HPO results:\n{df.to_string(index=False)}')
    print(f'  [{model_name}] Best: wd={best_wd:.0e}  '
          f'val_loss={df.iloc[0]["val_loss"]:.5f}\n')
    df.to_csv(os.path.join(fold_dir, f'{tag}_hpo_results.csv'), index=False)
    gc.collect()
    return best_wd

# FINAL TRAINING  (sharded, year-filtered, rescaled — mirrors train_nn_sharded)

In [ ]:
def train_nn_sharded_filtered_both(zl_cfg, mn_cfg, sources, feat_indices, rpi_abs_idx,
                                    rescale_a, rescale_b, cont_feat_pos,
                                    zl_save_path, mn_save_path, verbose=True):
    """
    Train LogReg and MortgageNet simultaneously on each chunk.
    Chunk is loaded once, both models trained on it, then freed.
    Halves I/O vs training sequentially.
    """
    rng = np.random.default_rng(SEED)
    bs  = zl_cfg['batch_size']

    zl_model = MortgageNet(
        input_dim=len(feat_indices), depth=0, hidden_dim=1,
        dropout_rate=0.0, norm_type='none').to(DEVICE)
    mn_model = MortgageNet(
        input_dim=len(feat_indices), depth=mn_cfg['depth'],
        hidden_dim=mn_cfg['hidden_dim'], dropout_rate=mn_cfg['dropout'],
        norm_type=mn_cfg['norm_type']).to(DEVICE)

    zl_opt = make_optimizer(zl_model, zl_cfg['lr'], zl_cfg['weight_decay'])
    mn_opt = make_optimizer(mn_model, mn_cfg['lr'], mn_cfg['weight_decay'])
    crit   = nn.CrossEntropyLoss()

    prefetch = FilteredChunkPrefetcher(
        feat_indices, rpi_abs_idx, rescale_a, rescale_b, cont_feat_pos)

    zl_history = {'tr_loss': []}
    mn_history = {'tr_loss': []}

    print('  Caching source sizes ...')
    src_sizes = []
    for path, yr in sources:
        with h5py.File(path, 'r') as f:
            src_sizes.append(f['y'].shape[0])
        print(f'    {os.path.basename(path):45s} '
              f'{src_sizes[-1]:>12,} rows  '
              f'(filter: {sorted(yr) if yr else "none"})')
    print(f'  Total stored rows: {sum(src_sizes):,}')

    n_epochs = zl_cfg['epochs']

    for epoch in range(1, n_epochs + 1):
        zl_total = mn_total = n_total = 0
        t_wait_total = t_train_total = 0.0
        ep_t0  = time.time()

        chunks = []
        for si in range(len(sources)): # ← Changed from order to sequential si
            path, yr = sources[si]
            for start in range(0, src_sizes[si], CHUNK_ROWS):
                chunks.append((path, start,
                                min(start + CHUNK_ROWS, src_sizes[si]), yr))

        # Global shuffle — interleaves 2021 chunks among 1999-2020 chunks
        rng.shuffle(chunks) # ← Added this line

        prefetch.start(*chunks[0])

        for ci, (path, start, end, yr) in enumerate(chunks):
            t0 = time.time()
            X_np, y_np = prefetch.get()
            t_get = time.time() - t0
            t_wait_total += t_get

            if ci + 1 < len(chunks):
                prefetch.start(*chunks[ci + 1])

            if len(y_np) == 0:
                print(f'    [chunk {ci+1:>2d}/{len(chunks)}] '
                      f'empty after year filter — skip', flush=True)
                continue

            X_tr = torch.from_numpy(X_np).to(DEVICE)
            y_tr = torch.from_numpy(y_np).to(DEVICE)
            del X_np, y_np; gc.collect()

            n_chunk = X_tr.shape[0]
            perm    = torch.randperm(n_chunk, device=DEVICE)

            t0 = time.time()

            # ── Train LogReg on this chunk ──────────────────────────────────
            zl_model.train()
            for i in range(0, n_chunk - bs + 1, bs):
                idx = perm[i:i+bs]
                X_b = X_tr[idx]; y_b = y_tr[idx]
                zl_opt.zero_grad()
                with torch.autocast(device_type='cuda', dtype=AMP_DTYPE,
                                    enabled=DEVICE.type == 'cuda'):
                    logits = zl_model(X_b)
                    loss   = crit(logits, y_b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(zl_model.parameters(), 1.0)
                zl_opt.step()
                zl_total += loss.item() * bs

            # ── Train MortgageNet on same chunk (same perm) ─────────────────
            mn_model.train()
            for i in range(0, n_chunk - bs + 1, bs):
                idx = perm[i:i+bs]
                X_b = X_tr[idx]; y_b = y_tr[idx]
                mn_opt.zero_grad()
                with torch.autocast(device_type='cuda', dtype=AMP_DTYPE,
                                    enabled=DEVICE.type == 'cuda'):
                    logits = mn_model(X_b)
                    loss   = crit(logits, y_b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(mn_model.parameters(), 1.0)
                mn_opt.step()
                mn_total += loss.item() * bs

            n_total += (n_chunk // bs) * bs
            t_train  = time.time() - t0
            t_train_total += t_train

            t0 = time.time()
            del X_tr, y_tr, perm; gc.collect()
            t_gc = time.time() - t0

            t0 = time.time()
            torch.cuda.empty_cache()
            t_cache = time.time() - t0

            print(f'    [chunk {ci+1:>2d}/{len(chunks)}] '
                  f'rows={n_chunk:>9,}  get={t_get:.2f}s  '
                  f'train={t_train:.1f}s  gc={t_gc:.2f}s  '
                  f'cache={t_cache:.2f}s  ({os.path.basename(path)})',
                  flush=True)

        zl_loss = zl_total / n_total if n_total > 0 else float('nan')
        mn_loss = mn_total / n_total if n_total > 0 else float('nan')
        zl_history['tr_loss'].append(zl_loss)
        mn_history['tr_loss'].append(mn_loss)

        ep_t = time.time() - ep_t0
        if verbose:
            print(f'  === Epoch {epoch:2d}/{n_epochs} | '
                  f'ZL_loss={zl_loss:.4f}  MN_loss={mn_loss:.4f} | '
                  f'{ep_t:.0f}s '
                  f'[wait={t_wait_total:.0f}s  train={t_train_total:.0f}s] ===',
                  flush=True)

    torch.save(zl_model.state_dict(), zl_save_path)
    torch.save(mn_model.state_dict(), mn_save_path)

    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    return zl_model, mn_model, zl_history, mn_history

# EVALUATION  (chunked streaming, year-filtered, rescaled)

In [ ]:
def evaluate_chunked_filtered_both(sources, feat_indices, rpi_abs_idx,
                                    rescale_a, rescale_b, cont_feat_pos,
                                    zl_model, mn_model, crit):
    """
    Evaluate LogReg and MortgageNet in a single streaming pass.
    Chunk loaded once, both models run forward, then chunk freed.
    Halves I/O vs evaluating sequentially.
    """
    zl_model.eval(); mn_model.eval()

    zl_loss = zl_correct = 0
    mn_loss = mn_correct = 0
    n = 0

    zl_preds_lst  = []; zl_labels_lst = []; zl_probs_lst  = []
    mn_preds_lst  = []; mn_labels_lst = []; mn_probs_lst  = []

    for h5_path, year_set in sources:
        with h5py.File(h5_path, 'r') as f:
            n_rows = f['X'].shape[0]

        for start in range(0, n_rows, CHUNK_ROWS):
            end = min(start + CHUNK_ROWS, n_rows)

            with h5py.File(h5_path, 'r') as f:
                X_raw   = f['X'][start:end]
                y_chunk = f['y'][start:end].astype(np.int64)

            if year_set is not None:
                rpi  = (X_raw[:, rpi_abs_idx].astype(np.int64)) // 100
                mask = np.isin(rpi, np.array(sorted(year_set)))
                X_raw   = X_raw[mask]
                y_chunk = y_chunk[mask]

            if len(y_chunk) == 0:
                del X_raw; continue

            X_chunk = X_raw[:, feat_indices].astype(np.float32)
            del X_raw                               # crucial RAM saver
            X_chunk[:, cont_feat_pos] = (
                X_chunk[:, cont_feat_pos].astype(np.float64) * rescale_a + rescale_b
            ).astype(np.float32)

            X_gpu = torch.from_numpy(X_chunk).to(DEVICE)
            y_gpu = torch.from_numpy(y_chunk).to(DEVICE)
            del X_chunk, y_chunk

            with torch.no_grad():
                for j in range(0, len(X_gpu), 32768):
                    X_b = X_gpu[j:j+32768]
                    y_b = y_gpu[j:j+32768]

                    # ── LogReg forward ──────────────────────────────────────
                    zl_logits = zl_model(X_b)
                    zl_l      = crit(zl_logits, y_b)
                    zl_preds  = zl_logits.argmax(dim=1)
                    zl_probs  = F.softmax(zl_logits, dim=1)
                    zl_loss    += zl_l.item() * len(y_b)
                    zl_correct += (zl_preds == y_b).sum().item()
                    zl_preds_lst.append(zl_preds.cpu())
                    zl_labels_lst.append(y_b.cpu())
                    zl_probs_lst.append(zl_probs.cpu())

                    # ── MortgageNet forward ─────────────────────────────────
                    mn_logits = mn_model(X_b)
                    mn_l      = crit(mn_logits, y_b)
                    mn_preds  = mn_logits.argmax(dim=1)
                    mn_probs  = F.softmax(mn_logits, dim=1)
                    mn_loss    += mn_l.item() * len(y_b)
                    mn_correct += (mn_preds == y_b).sum().item()
                    mn_preds_lst.append(mn_preds.cpu())
                    mn_labels_lst.append(y_b.cpu())
                    mn_probs_lst.append(mn_probs.cpu())

                    n += len(y_b)

            del X_b, y_b, zl_logits, mn_logits

            del X_gpu, y_gpu
            gc.collect()
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

            print(f'    eval chunk {start:>12,}-{end:>12,} / {n_rows:,}  '
                  f'({os.path.basename(h5_path)})', flush=True)

    def _compute_metrics(preds_lst, labels_lst, probs_lst, correct_sum):
        all_preds  = torch.cat(preds_lst).numpy()
        all_labels = torch.cat(labels_lst).numpy()
        all_probs  = torch.cat(probs_lst).numpy()

        macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        accuracy  = correct_sum / n if n > 0 else 0.0

        CHUNK_LL = 10_000_000
        chunks_ll = [(s, min(s + CHUNK_LL, n)) for s in range(0, n, CHUNK_LL)]
        def _chunk_ll(args):
            s, e = args
            return log_loss(all_labels[s:e], all_probs[s:e],
                            labels=list(range(NUM_CLASSES))) * (e - s)
        with ThreadPoolExecutor(max_workers=n_cpus) as pool:
            ll_parts = list(pool.map(_chunk_ll, chunks_ll))
        neg_ll = sum(ll_parts) / n

        prec, rec, f1_pc, sup = precision_recall_fscore_support(
            all_labels, all_preds, labels=list(range(NUM_CLASSES)),
            average=None, zero_division=0)
        per_class_df = pd.DataFrame(
            {'precision': prec, 'recall': rec, 'f1': f1_pc,
             'support': sup.astype(int)},
            index=STATE_NAMES)

        del all_preds, all_labels, all_probs; gc.collect()
        scalar = dict(log_loss=neg_ll, macro_f1=macro_f1, accuracy=accuracy)
        return scalar, per_class_df

    zl_scalar, zl_pc = _compute_metrics(
        zl_preds_lst, zl_labels_lst, zl_probs_lst, zl_correct)
    mn_scalar, mn_pc = _compute_metrics(
        mn_preds_lst, mn_labels_lst, mn_probs_lst, mn_correct)

    ctypes.CDLL('libc.so.6').malloc_trim(0)
    return zl_scalar, zl_pc, mn_scalar, mn_pc

In [ ]:
def ckpt_load():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE) as f:
            c = json.load(f)
        print(f'[CKPT] Resumed: {list(c.keys())}')
        return c
    return {}

def ckpt_save(state):
    with open(CKPT_FILE, 'w') as f:
        json.dump(state, f, indent=2)
    print('[CKPT] Saved.', flush=True)

# MAIN

In [ ]:
try:
    if __name__ == '__main__':
        print('=' * 70)
        print('ROLLING TEST — 3-Fold Temporal Validation')
        print('Fold 1: Test 2016-2018 | Fold 2: Test 2019-2021 | Fold 3: Test 2022-2025')
        print(f'HPO: grid search WD_GRID={WD_GRID}  |  {HPO_EPOCHS} epochs/trial')
        print(f'     {HPO_ROWS//1_000_000:.0f}M train rows  |  '
              f'{HPO_VAL_ROWS//1_000_000:.0f}M val rows')
        print(f'Final training: {FINAL_EPOCHS} epochs, both models trained per chunk')
        print(f'OUTPUT: {OUTPUT_DIR}')
        print('=' * 70)

        # ── Feature columns and index maps ──────────────────────────────────
        with open(COLS_JSON) as fj:
            FEATURE_COLS = json.load(fj)
        MODEL_FEAT_IDX, CONT_HDF5_IDX, CONT_FEAT_POS, RPI_ABS_IDX = \
            build_feature_maps(FEATURE_COLS)
        N_FEATURES = len(MODEL_FEAT_IDX)
        print(f'\nHDF5 columns   : {len(FEATURE_COLS)}')
        print(f'Model features : {N_FEATURES}  (dropped: {REPORTING_PERIOD_COL})')
        print(f'Continuous cols: {len(CONT_HDF5_IDX)}')

        # ── Build fold definitions ───────────────────────────────────────────
        # test.h5 is read directly from Drive (year-filtered, chunked streaming)
        # No NVMe copy of test data — insufficient space after 10 shards
        FOLDS = _make_folds()

        # ── Existing scaler (fit on 1999-2016 by script2) ───────────────────
        # scaler.pkl is a dict with keys 'scaler' and 'continuous_indices'
        scaler_data     = joblib.load(SCALER_PKL)
        existing_scaler = scaler_data['scaler']
        # Override CONT_HDF5_IDX with the stored indices — guaranteed to match
        # exactly what was used when the scaler was originally fitted
        CONT_HDF5_IDX   = np.array(scaler_data['continuous_indices'])
        # Rebuild CONT_FEAT_POS using the corrected indices
        feat_pos_map    = {hdf5_idx: pos
                           for pos, hdf5_idx in enumerate(MODEL_FEAT_IDX)}
        CONT_FEAT_POS   = np.array([feat_pos_map[i] for i in CONT_HDF5_IDX])
        print(f'Existing scaler : {existing_scaler.mean_.shape[0]} continuous features')
        print(f'Continuous cols : {len(CONT_HDF5_IDX)} (loaded from scaler.pkl)')

        ckpt      = ckpt_load()
        crit_eval = nn.CrossEntropyLoss()

        # ══════════════════════════════════════════════════════════════════════
        # FOLD LOOP
        # ══════════════════════════════════════════════════════════════════════
        for fold in FOLDS:
            fname    = fold['name']
            test_lbl = fold['test_label']
            fold_dir = os.path.join(OUTPUT_DIR, fname)
            os.makedirs(fold_dir, exist_ok=True)
            fk = ckpt.setdefault(fname, {})

            print(f'\n{"=" * 70}')
            print(f'FOLD: {fname.upper()}  |  Test period: {test_lbl}')
            print(f'{"=" * 70}')

            # ── Step 1: Fit fold-specific scaler ─────────────────────────────
            if not fk.get('scaler_done'):
                print(f'\n[{fname}] 1/7 Fitting fold scaler on fold train rows...')
                fold_scaler = fit_fold_scaler(
                    fold['scaler_sources'], CONT_HDF5_IDX,
                    RPI_ABS_IDX, existing_scaler)
                joblib.dump(fold_scaler,
                            os.path.join(fold_dir, 'scaler_fold.pkl'))
                fk['scaler_done'] = True; ckpt_save(ckpt)
            else:
                print(f'[{fname}][CKPT] Scaler done — reloading...')
                fold_scaler = joblib.load(
                    os.path.join(fold_dir, 'scaler_fold.pkl'))

            rescale_a, rescale_b = compute_rescale_coeffs(
                existing_scaler, fold_scaler)
            print(f'  Rescale a: [{rescale_a.min():.4f}, {rescale_a.max():.4f}]  '
                  f'b: [{rescale_b.min():.4f}, {rescale_b.max():.4f}]')

            # ── Step 2: Load HPO data into VRAM ──────────────────────────────
            # Pushed once, reused across all 8 HPO trials (4×ZL + 4×MN)
            # VRAM budget: 60M×N_FEAT×4 + 15M×N_FEAT×4 + model ≈ fits A100 80GB
            _hpo_needed = (not fk.get('zl_hpo_done')) or (not fk.get('mn_hpo_done'))

            if _hpo_needed:
                print(f'\n[{fname}] 2/7 Loading HPO data into VRAM '
                      f'({HPO_ROWS//1_000_000:.0f}M train + '
                      f'{HPO_VAL_ROWS//1_000_000:.0f}M val)...')

                X_hpo_np, y_hpo_np = load_hpo_sample(
                    fold['hpo_train_sources'], MODEL_FEAT_IDX, RPI_ABS_IDX,
                    rescale_a, rescale_b, CONT_FEAT_POS, HPO_ROWS)
                X_val_np, y_val_np = load_hpo_sample(
                    fold['hpo_val_sources'], MODEL_FEAT_IDX, RPI_ABS_IDX,
                    rescale_a, rescale_b, CONT_FEAT_POS, HPO_VAL_ROWS)

                print('  Pushing HPO data to GPU...', flush=True)
                X_tr_gpu  = torch.from_numpy(X_hpo_np).to(DEVICE)
                y_tr_gpu  = torch.from_numpy(y_hpo_np).to(DEVICE)
                X_val_gpu = torch.from_numpy(X_val_np).to(DEVICE)
                y_val_gpu = torch.from_numpy(y_val_np).to(DEVICE)
                del X_hpo_np, y_hpo_np, X_val_np, y_val_np; gc.collect()

                if DEVICE.type == 'cuda':
                    used = torch.cuda.memory_allocated() / 1e9
                    tot  = torch.cuda.get_device_properties(0).total_memory / 1e9
                    print(f'  VRAM after HPO data push: {used:.1f} / {tot:.1f} GB')
            else:
                print(f'[{fname}][CKPT] Both HPOs done — skipping VRAM load')

            # ── Step 3: LogReg HPO (grid search) ─────────────────────────────
            if not fk.get('zl_hpo_done'):
                print(f'\n[{fname}] 3/7 LogReg HPO (grid search, '
                      f'{len(WD_GRID)} trials)...')
                zl_base = dict(depth=0, hidden_dim=1, dropout=0.0,
                               norm_type='none', lr=ZL_LR)
                best_wd_zl = run_hpo_grid(
                    'Logistic Regression', zl_base,
                    X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu,
                    WD_GRID, fold_dir)
                fk['zl_hpo_done'] = True
                fk['best_wd_zl']  = best_wd_zl
                ckpt_save(ckpt)
            else:
                best_wd_zl = fk['best_wd_zl']
                print(f'[{fname}][CKPT] ZL HPO done — best_wd={best_wd_zl:.0e}')

            # ── Step 4: MortgageNet HPO (grid search, fixed arch) ────────────
            if not fk.get('mn_hpo_done'):
                print(f'\n[{fname}] 4/7 MortgageNet HPO (grid search, '
                      f'{len(WD_GRID)} trials, arch={MN_ARCH})...')
                mn_base = dict(**MN_ARCH, lr=MN_LR)
                best_wd_mn = run_hpo_grid(
                    'MortgageNet', mn_base,
                    X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu,
                    WD_GRID, fold_dir)
                fk['mn_hpo_done'] = True
                fk['best_wd_mn']  = best_wd_mn
                ckpt_save(ckpt)
            else:
                best_wd_mn = fk['best_wd_mn']
                print(f'[{fname}][CKPT] MN HPO done — best_wd={best_wd_mn:.2e}')

            # ── Step 5: Free HPO VRAM before final training ──────────────────
            # Final training needs ~41 GB per chunk; HPO data (~51 GB) must go first
            if _hpo_needed:
                print(f'\n  Freeing HPO GPU tensors...')
                del X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu
                gc.collect(); torch.cuda.empty_cache()
                if DEVICE.type == 'cuda':
                    print(f'  VRAM after free: '
                          f'{torch.cuda.memory_allocated() / 1e9:.1f} GB')

            # ── Step 6: Joint final training — LogReg + MortgageNet ──────────
            # Both models trained on each chunk simultaneously — halves I/O
            ZL_CFG  = dict(depth=0, hidden_dim=1, dropout=0.0, norm_type='none',
                           lr=ZL_LR, weight_decay=best_wd_zl,
                           batch_size=16384, epochs=FINAL_EPOCHS)
            MN_CFG  = dict(**MN_ARCH, lr=MN_LR, weight_decay=best_wd_mn,
                           batch_size=16384, epochs=FINAL_EPOCHS)
            zl_path = os.path.join(fold_dir, 'logreg.pt')
            mn_path = os.path.join(fold_dir, 'mortgagenet.pt')
            zl_model = mn_model = None   # safe default for cleanup

            if not fk.get('joint_train_done'):
                print(f'\n[{fname}] 6/7 Joint final training — LogReg + MortgageNet '
                      f'({FINAL_EPOCHS} epochs, both models trained per chunk)...')
                json.dump(ZL_CFG,
                          open(os.path.join(fold_dir, 'zl_cfg.json'), 'w'),
                          indent=2)
                json.dump(MN_CFG,
                          open(os.path.join(fold_dir, 'mn_cfg.json'), 'w'),
                          indent=2)
                zl_model, mn_model, _, _ = train_nn_sharded_filtered_both(
                    ZL_CFG, MN_CFG, fold['final_train_src'], MODEL_FEAT_IDX,
                    RPI_ABS_IDX, rescale_a, rescale_b, CONT_FEAT_POS,
                    zl_path, mn_path)
                torch.save({'model_state': zl_model.state_dict(),
                            'cfg':         ZL_CFG,
                            'feat_idx':    MODEL_FEAT_IDX}, zl_path)
                torch.save({'model_state': mn_model.state_dict(),
                            'cfg':         MN_CFG,
                            'feat_idx':    MODEL_FEAT_IDX}, mn_path)
                fk['joint_train_done'] = True; ckpt_save(ckpt)
            else:
                print(f'[{fname}][CKPT] Joint training done — skipping retrain.')
                if not fk.get('joint_eval_done'):
                    print(f'  Reloading both models for evaluation...')
                    zl_model = MortgageNet(input_dim=N_FEATURES, depth=0,
                                           hidden_dim=1, dropout_rate=0.0,
                                           norm_type='none').to(DEVICE)
                    mn_model = MortgageNet(input_dim=N_FEATURES,
                       depth=MN_ARCH['depth'],
                       hidden_dim=MN_ARCH['hidden_dim'],
                       dropout_rate=MN_ARCH['dropout'],
                       norm_type=MN_ARCH['norm_type']).to(DEVICE)
                    zl_model.load_state_dict(
                        torch.load(zl_path, map_location=DEVICE,
                                   weights_only=True)['model_state'])
                    mn_model.load_state_dict(
                        torch.load(mn_path, map_location=DEVICE,
                                   weights_only=True)['model_state'])
                    zl_model.eval(); mn_model.eval()
                    gc.collect()
                    torch.cuda.synchronize()
                    torch.cuda.empty_cache()

            # ── Step 7: Joint evaluation — LogReg + MortgageNet ──────────────
            # Both models evaluated on each test chunk simultaneously — halves I/O
            if not fk.get('joint_eval_done'):
                print(f'\n[{fname}] 7/7 Joint evaluation on test ({test_lbl})...')
                zl_scalar, zl_pc, mn_scalar, mn_pc = evaluate_chunked_filtered_both(
                    fold['test_sources'], MODEL_FEAT_IDX, RPI_ABS_IDX,
                    rescale_a, rescale_b, CONT_FEAT_POS,
                    zl_model, mn_model, crit_eval)

                print(f'\n  LogReg:      log_loss={zl_scalar["log_loss"]:.5f}  '
                      f'macro_f1={zl_scalar["macro_f1"]:.5f}  '
                      f'accuracy={zl_scalar["accuracy"]:.5f}')
                print(f'  MortgageNet: log_loss={mn_scalar["log_loss"]:.5f}  '
                      f'macro_f1={mn_scalar["macro_f1"]:.5f}  '
                      f'accuracy={mn_scalar["accuracy"]:.5f}')
                print(f'  ZL per-class:\n{zl_pc.round(4).to_string()}')
                print(f'  MN per-class:\n{mn_pc.round(4).to_string()}')

                zl_pc.to_csv(os.path.join(fold_dir, 'logreg_per_class.csv'))
                mn_pc.to_csv(os.path.join(fold_dir, 'mortgagenet_per_class.csv'))
                pd.DataFrame([{**zl_scalar, 'fold': fname,
                               'test_period': test_lbl,
                               'model': 'Logistic Regression'}]).to_csv(
                    os.path.join(fold_dir, 'logreg_scalar.csv'), index=False)
                pd.DataFrame([{**mn_scalar, 'fold': fname,
                               'test_period': test_lbl,
                               'model': 'MortgageNet'}]).to_csv(
                    os.path.join(fold_dir, 'mortgagenet_scalar.csv'), index=False)
                fk['joint_eval_done'] = True; ckpt_save(ckpt)
            else:
                print(f'[{fname}][CKPT] Joint eval done — loading from CSVs...')
                zl_scalar = pd.read_csv(
                    os.path.join(fold_dir, 'logreg_scalar.csv')).iloc[0].to_dict()
                mn_scalar = pd.read_csv(
                    os.path.join(fold_dir, 'mortgagenet_scalar.csv')).iloc[0].to_dict()

            # ── OOM cleanup after fold ────────────────────────────────────────
            if zl_model is not None: del zl_model
            if mn_model is not None: del mn_model
            gc.collect(); torch.cuda.empty_cache()
            ctypes.CDLL('libc.so.6').malloc_trim(0)

            fk['fold_done'] = True; ckpt_save(ckpt)
            print(f'\n[{fname}] ✓ COMPLETE')

        # ══════════════════════════════════════════════════════════════════════
        # SUMMARY TABLE
        # ══════════════════════════════════════════════════════════════════════
        print('\n' + '=' * 70)
        print('ROLLING TEST — FINAL SUMMARY')
        print('=' * 70)

        all_rows = []
        for fold in FOLDS:
            fold_dir = os.path.join(OUTPUT_DIR, fold['name'])
            for model_name, csv_name in [
                ('Logistic Regression', 'logreg_scalar.csv'),
                ('MortgageNet',         'mortgagenet_scalar.csv'),
            ]:
                csv_path = os.path.join(fold_dir, csv_name)
                if os.path.exists(csv_path):
                    row = pd.read_csv(csv_path).iloc[0].to_dict()
                    row.setdefault('fold',        fold['name'])
                    row.setdefault('test_period', fold['test_label'])
                    row.setdefault('model',       model_name)
                    all_rows.append(row)
                else:
                    print(f'  [WARN] Missing: {csv_path}')

        summary_df = pd.DataFrame(all_rows)
        cols = ['fold', 'test_period', 'model', 'log_loss', 'macro_f1', 'accuracy']
        print('\n' + summary_df[cols].round(5).to_string(index=False))
        summary_df.to_csv(os.path.join(OUTPUT_DIR, 'rolling_summary.csv'),
                          index=False)

        # Ordering check per fold
        print('\nPerformance ordering check (MortgageNet log_loss < LogReg):')
        for fold in FOLDS:
            sub = summary_df[summary_df['fold'] == fold['name']]
            if len(sub) == 2:
                mn_ll = float(
                    sub[sub['model'] == 'MortgageNet']['log_loss'].iloc[0])
                zl_ll = float(
                    sub[sub['model'] == 'Logistic Regression']['log_loss'].iloc[0])
                ok = mn_ll < zl_ll
                print(f'  {fold["name"]} ({fold["test_label"]}): '
                      f'MN={mn_ll:.5f}  ZL={zl_ll:.5f}  '
                      f'→ {"CONFIRMED" if ok else "NOT MET"}')

        # ── Per-class summary for key states ──────────────────────────────────
        print('\n=== PER-CLASS SUMMARY — KEY STATES ===')
        key_states = ['D90+', 'Foreclosure', 'Current']
        for fold in FOLDS:
            fold_dir = os.path.join(OUTPUT_DIR, fold['name'])
            print(f'\n{fold["name"]} ({fold["test_label"]}):')
            for model_name, csv_name in [
                ('LogReg',      'logreg_per_class.csv'),
                ('MortgageNet', 'mortgagenet_per_class.csv'),
            ]:
                csv_path = os.path.join(fold_dir, csv_name)
                if os.path.exists(csv_path):
                    df = pd.read_csv(csv_path, index_col=0)
                    print(f'  {model_name}:')
                    for state in key_states:
                        if state in df.index:
                            row = df.loc[state]
                            print(f'    {state:<12} '
                                  f'precision={row["precision"]:.4f}  '
                                  f'recall={row["recall"]:.4f}  '
                                  f'f1={row["f1"]:.4f}  '
                                  f'support={int(row["support"]):,}')

        print(f'\nAll outputs saved to: {OUTPUT_DIR}')
        print('Rolling test complete.')

except Exception as _e:
    import traceback
    _err = traceback.format_exc()
    _crash = os.path.join(OUTPUT_DIR, 'CRASH_LOG.txt')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(_crash, 'w') as _f:
        _f.write(_err)
    print(f'CRASHED — log saved to {_crash}\n{_err}', flush=True)

finally:
    print('Releasing runtime to stop compute billing...', flush=True)
    from google.colab import runtime
    runtime.unassign()

ROLLING TEST — 3-Fold Temporal Validation
Fold 1: Test 2016-2018 | Fold 2: Test 2019-2021 | Fold 3: Test 2022-2025
HPO: grid search WD_GRID=[1e-05, 0.0001, 0.001, 0.01]  |  10 epochs/trial
     5M train rows  |  1M val rows
Final training: 10 epochs, both models trained per chunk
OUTPUT: /content/drive/MyDrive/dissertation/models/rolling_test_sample_v2

HDF5 columns   : 175
Model features : 174  (dropped: Reporting_Period_Int)
Continuous cols: 21
Existing scaler : 21 continuous features
Continuous cols : 21 (loaded from scaler.pkl)

FOLD: FOLD3  |  Test period: 2022-2025

[fold3] 1/7 Fitting fold scaler on fold train rows...
  [scaler] train_shard_0_merged.h5: 5,178,328 rows
  [scaler] train_shard_1_merged.h5: 5,362,677 rows
  [scaler] train_shard_2_merged.h5: 5,166,729 rows
  [scaler] train_shard_3_merged.h5: 5,162,163 rows
  [scaler] train_shard_4_merged.h5: 5,188,305 rows
  [scaler] train_shard_5_merged.h5: 5,182,786 rows
  [scaler] train_shard_6_merged.h5: 5,129,118 rows
  [scaler]

In [ ]:
# ============================================================
# verify_test_2021.py
# Confirm test_2021.h5 contains ONLY year 2021 rows.
# Standalone script — mounts Drive, installs deps fresh.
# ============================================================

# ── 1. Mount Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ── 2. Install ───────────────────────────────────────────────
!pip install hdf5plugin --quiet

import os
import numpy as np
os.environ['HDF5_PLUGIN_MAX_THREADS'] = str(os.cpu_count())
import hdf5plugin   # MUST import before h5py
import h5py

# ── 3. Path ──────────────────────────────────────────────────
path    = '/content/drive/MyDrive/dissertation/hdf5_full_merged/test_2021.h5'
RPI_IDX = 13

print('=' * 60)
print('VERIFY test_2021.h5 — full scan')
print('=' * 60)

with h5py.File(path, 'r') as f:
    n_rows = f['X'].shape[0]
    n_cols = f['X'].shape[1]
    keys   = list(f.keys())
    print(f'Rows: {n_rows:,}  Cols: {n_cols}  Keys: {keys}')

    # Read the RPI column in chunks to avoid loading huge X unnecessarily
    # (we only need column RPI_IDX, but row-oriented HDF5 means we still
    #  read full row-chunks under the hood — that's fine, still far
    #  smaller than the full X load since file itself is much smaller now)
    CHUNK = 20_000_000
    years_found = set()
    for start in range(0, n_rows, CHUNK):
        end = min(start + CHUNK, n_rows)
        rpi_chunk = (f['X'][start:end, RPI_IDX].astype(np.int64)) // 100
        years_found.update(rpi_chunk.tolist())
        print(f'  scanned {start:,}-{end:,} / {n_rows:,}', flush=True)

years_found = sorted(years_found)
print(f'\nYears found across ALL {n_rows:,} rows: {years_found}')

if years_found == [2021]:
    print('\n✓ CONFIRMED: file contains ONLY year 2021 data.')
else:
    print(f'\n✗ WARNING: unexpected years present: {years_found}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 61.4 MB/s eta 0:00:00
VERIFY test_2021.h5 — full scan
Rows: 142,640,611  Cols: 175  Keys: ['X', 'y']
  scanned 0-20,000,000 / 142,640,611
  scanned 20,000,000-40,000,000 / 142,640,611
  scanned 40,000,000-60,000,000 / 142,640,611
  scanned 60,000,000-80,000,000 / 142,640,611
  scanned 80,000,000-100,000,000 / 142,640,611
  scanned 100,000,000-120,000,000 / 142,640,611
  scanned 120,000,000-140,000,000 / 142,640,611
  scanned 140,000,000-142,640,611 / 142,640,611

Years found across ALL 142,640,611 rows: [2021]

✓ CONFIRMED: file contains ONLY year 2021 data.
